# Astronomical Data

Generates a CSV with **illuminance**, **sun/moon position**, **moon phase**, and 
**percent darkness** (Helfenstein & Veverka 1987 / Hapke 1986 disk-integrated model).

Prints `illuminance_lux` for each row across a configurable list of dates.

---

### Percent Darkness Method
Percent darkness is estimated using the lunar photometric model of Helfenstein and Veverka (1987), 
which integrates Hapke's (1986) equation fit to the lunar disk-integrated visual lightcurve. 
Whole-disk brightness is calculated for each phase angle (0° = full moon, 180° = new moon). 
The brightness at 2° phase is assigned **0% darkness** and at 179° phase **100% darkness**, 
yielding a phase curve across the full lunar cycle.

---

### Requirements
```bash
pip install astropy pytz pandas timezonefinder numpy
```


In [8]:
import math
import numpy as np
import pytz
import pandas as pd
from datetime import datetime, timedelta
from astropy.coordinates import AltAz, EarthLocation, get_body, get_sun
from astropy.time import Time
import astropy.units as u
from timezonefinder import TimezoneFinder

print('✓ Imports OK')

✓ Imports OK


### Configuration
Edit the values below, then run all cells.

In [9]:
LAT      =  9.338212
LON      = -82.258937

DATES    = [
    '2018-09-27',
    '2018-09-29',
    '2018-10-01',
    '2018-10-03',
    '2018-10-05',
    '2018-10-08',
    '2022-07-14',
    '2022-07-16',
    '2022-07-18',
    '2022-07-20',
    '2022-07-22',
    '2022-07-24'
]
DAY_START = '17:00'   # local time start applied to each date  (HH:MM)
DAY_END   = '20:00'   # local time end applied to each date    (HH:MM)
INTERVAL  = 1         # minutes between rows
OUTPUT    = 'PA_NH_Astronomical_Data.csv'

# Timezone: auto-detected from coordinates, or set manually.
TIMEZONE = TimezoneFinder().timezone_at(lat=LAT, lng=LON)
# TIMEZONE = 'America/New_York'

print(f'Timezone  : {TIMEZONE}')
print(f'Location  : {LAT}°N, {LON}°E')
print(f'Dates     : {DATES}')
print(f'Window    : {DAY_START}  →  {DAY_END}  (every {INTERVAL} min)')


Timezone  : America/Panama
Location  : 9.338212°N, -82.258937°E
Dates     : ['2018-09-27', '2018-09-29', '2018-10-01', '2018-10-03', '2018-10-05', '2018-10-08', '2022-07-14', '2022-07-16', '2022-07-18', '2022-07-20', '2022-07-22', '2022-07-24']
Window    : 17:00  →  20:00  (every 1 min)


### Helfenstein & Veverka (1987) / Hapke (1986) Percent-Darkness Model

Disk-integrated parameters from H&V (1987) Table II: `w=0.21, h=0.07, S(0)=0.71, b=0.29, c=0.39, θ̄=20°`

- **Phase angle convention:** 0° = full moon, 180° = new moon  
- **0% darkness** = brightness at 2° phase (smallest phase visible from Earth)  
- **100% darkness** = brightness at 179° phase  


In [10]:
# H&V (1987) Table II disk-integrated parameters
_w, _h, _S0, _b, _c, _tbar = 0.21, 0.07, 0.71, 0.29, 0.39, 20.0

def _hapke_brightness(alpha_deg):
    """Whole-disk brightness from Hapke (1986) using H&V (1987) Table II params."""
    alpha_deg = max(0.001, min(alpha_deg, 179.999))
    a = math.radians(alpha_deg)
    phase_fn   = 1 + _b * math.cos(a) + _c * (1.5 * math.cos(a) ** 2 - 0.5)
    opposition = _S0 / (1 + (1 / _h) * math.tan(a / 2))
    roughness  = 1 - 0.104 * math.sin(math.radians(_tbar)) * (1 - math.exp(-3.0 * a))
    t1 = (_w / 8) * ((1 + opposition) * phase_fn - 1)
    t2 = (2 * _w) / (3 * math.pi) * ((math.pi - a) * math.cos(a) + math.sin(a))
    return (t1 + t2) * roughness

_B_FULL = _hapke_brightness(2)    # 0%  darkness reference
_B_NEW  = _hapke_brightness(179)  # 100% darkness reference

def percent_darkness(alpha_deg):
    """Percent darkness from phase angle via H&V (1987) phase curve."""
    B = _hapke_brightness(alpha_deg)
    return round(min(100.0, max(0.0, 100 * (_B_FULL - B) / (_B_FULL - _B_NEW))), 2)

print('H&V (1987) / Hapke (1986) model ready')
print(f'  Full moon reference  ( 2° phase) : {_B_FULL:.6f}')
print(f'  New moon reference   (179° phase): {_B_NEW:.6f}')


H&V (1987) / Hapke (1986) model ready
  Full moon reference  ( 2° phase) : 0.182144
  New moon reference   (179° phase): 0.002540


### Illuminance & Phase Helpers

In [11]:
def _sun_lux(alt):
    """Approximate horizontal illuminance (lux) from solar altitude (degrees)."""
    if alt >= 0:    return 133_775 * math.sin(math.radians(max(alt, 0))) ** 0.833
    if alt >= -6:   return max(0.0, 3.4   * (6  + alt))   # civil twilight
    if alt >= -12:  return max(0.0, 0.1   * (12 + alt))   # nautical twilight
    if alt >= -18:  return max(0.0, 0.001 * (18 + alt))   # astronomical twilight
    return 0.0

def _moon_lux(moon_alt, phase_angle_deg):
    if moon_alt <= 0:
        return 0.0
    hapke_scale = _hapke_brightness(phase_angle_deg) / _B_FULL
    return max(0.0, 0.27 * hapke_scale * math.sin(math.radians(moon_alt)))

def _phase_fraction(phase_angle_deg):
    """Illuminated fraction of the moon disk from phase angle."""
    return round((1 - math.cos(math.radians(180.0 - phase_angle_deg))) / 2, 4)

def _phase_name(frac):
    if frac <  0.01: return 'New Moon'
    if frac <  0.24: return 'Waxing Crescent'
    if frac <  0.26: return 'First Quarter'
    if frac <  0.49: return 'Waxing Gibbous'
    if frac <  0.51: return 'Half Moon'
    if frac <  0.74: return 'Waning Gibbous'
    if frac <  0.76: return 'Last Quarter'
    if frac <  0.99: return 'Waning Crescent'
    return 'Full Moon'

def _light_regime(sun_alt, moon_alt):
    if sun_alt >    0: return 'Daylit'
    if sun_alt >=  -6: return 'Civil Twilight'
    if sun_alt >= -12: return 'Nautical Twilight'
    if sun_alt >= -18: return 'Astronomical Twilight'
    if moon_alt >   0: return 'Moonlit'
    return 'Moonless Night'

print('✓ Helpers defined')

✓ Helpers defined


### Data Generation

In [12]:
def generate_sky_data(lat, lon, start_local, end_local, interval_min, tz_str):
    tz       = pytz.timezone(tz_str)
    location = EarthLocation(lat=lat * u.deg, lon=lon * u.deg)
    rows     = []
    current  = start_local

    while current <= end_local:
        utc_dt   = tz.localize(current).astimezone(pytz.utc)
        obs_time = Time(utc_dt)
        frame    = AltAz(obstime=obs_time, location=location)

        sun      = get_sun(obs_time)
        sun_aa   = sun.transform_to(frame)
        moon_aa  = get_body('moon', obs_time, location).transform_to(frame)

        # Geocentric phase angle — avoids coordinate-transform warnings
        moon_geo  = get_body('moon', obs_time)
        phase_deg = 180.0 - sun.separation(moon_geo).deg
        pfrac     = _phase_fraction(phase_deg)

        rows.append({
            'date':                current.strftime('%Y-%m-%d'),
            'time':                current.strftime('%H:%M:%S'),
            'illuminance_lux': round(_sun_lux(sun_aa.alt.deg) + _moon_lux(moon_aa.alt.deg, phase_deg), 4),
            'light_regime':        _light_regime(sun_aa.alt.deg, moon_aa.alt.deg),
            'sun_altitude_deg':    round(sun_aa.alt.deg,  4),
            'sun_azimuth_deg':     round(sun_aa.az.deg,   4),
            'moon_altitude_deg':   round(moon_aa.alt.deg, 4),
            'moon_azimuth_deg':    round(moon_aa.az.deg,  4),
            'phase_angle_deg':     round(phase_deg, 4),
            'pct_darkness':        percent_darkness(phase_deg),
            'moon_phase_fraction': pfrac,
            'moon_phase_name':     _phase_name(pfrac),
        })
        current += timedelta(minutes=interval_min)

    return pd.DataFrame(rows)



### Run

In [13]:
fmt = '%Y-%m-%d %H:%M'
all_dfs = []
civil_twilight_rows = []

for date_str in DATES:
    start_dt = datetime.strptime(f'{date_str} {DAY_START}', fmt)
    end_dt   = datetime.strptime(f'{date_str} {DAY_END}',   fmt)

    df = generate_sky_data(
        lat=LAT, lon=LON,
        start_local=start_dt,
        end_local=end_dt,
        interval_min=INTERVAL,
        tz_str=TIMEZONE,
    )
    all_dfs.append(df)

    print(f'\n── {date_str} ──────────────────────────────')
    print(df[['date', 'time', 'illuminance_lux']].to_string(index=False))

# Create datetime column
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])

# First Civil Twilight row after noon
civil_rows = df[
    (df["light_regime"] == "Civil Twilight") &
    (df["datetime"].dt.hour >= 12)
]

if not civil_rows.empty:

    first_civil = civil_rows.iloc[0]

    target_time = first_civil["datetime"] + pd.Timedelta(minutes=45)

    # Find row closest to 45 min after civil twilight begins
    idx = (df["datetime"] - target_time).abs().idxmin()

    sample_row = df.loc[idx]

    civil_twilight_rows.append({
        "date": sample_row["date"],
        "civil_twilight_start":
            first_civil["datetime"].strftime("%Y-%m-%d %H:%M:%S"),
        "sample_time":
            sample_row["datetime"].strftime("%Y-%m-%d %H:%M:%S"),
        "illuminance_lux":
            sample_row["illuminance_lux"]
    })
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df.to_csv(OUTPUT, index=False)
print(f'\n✅ Saved {len(combined_df)} rows → {OUTPUT}')

civil_df = pd.DataFrame(civil_twilight_rows)

civil_df.to_csv(
    "BZ_Illuminance_45min_After_Civil_Twilight.csv",
    index=False
)

print(
    f"Saved {len(civil_df)} rows to "
    "BZ_Illuminance_45min_After_Civil_Twilight.csv"
)


── 2018-09-27 ──────────────────────────────
      date     time  illuminance_lux
2018-09-27 17:00:00       53318.2587
2018-09-27 17:01:00       52775.0576
2018-09-27 17:02:00       52229.8807
2018-09-27 17:03:00       51682.7167
2018-09-27 17:04:00       51133.5536
2018-09-27 17:05:00       50582.3788
2018-09-27 17:06:00       50029.1795
2018-09-27 17:07:00       49473.9418
2018-09-27 17:08:00       48916.6517
2018-09-27 17:09:00       48357.2942
2018-09-27 17:10:00       47795.8538
2018-09-27 17:11:00       47232.3141
2018-09-27 17:12:00       46666.6583
2018-09-27 17:13:00       46098.8684
2018-09-27 17:14:00       45528.9259
2018-09-27 17:15:00       44956.8113
2018-09-27 17:16:00       44382.5043
2018-09-27 17:17:00       43805.9835
2018-09-27 17:18:00       43227.2266
2018-09-27 17:19:00       42646.2102
2018-09-27 17:20:00       42062.9099
2018-09-27 17:21:00       41477.3000
2018-09-27 17:22:00       40889.3536
2018-09-27 17:23:00       40299.0425
2018-09-27 17:24:00       397

### Data Preview

In [14]:
combined_df.head(10)


,date,time,illuminance_lux,light_regime,sun_altitude_deg,sun_azimuth_deg,moon_altitude_deg,moon_azimuth_deg,phase_angle_deg,pct_darkness,moon_phase_fraction,moon_phase_name,datetime
0,2018-09-27,17:00:00,53318.2587,Daylit,19.3564,264.6600,-49.2657,64.0547,33.6645,27.72,0.9161,Waning Crescent,NaT
1,2018-09-27,17:01:00,52775.0576,Daylit,19.1107,264.7082,-49.0522,64.2079,33.6729,27.73,0.9161,Waning Crescent,NaT
2,2018-09-27,17:02:00,52229.8807,Daylit,18.8649,264.7563,-48.8386,64.3595,33.6812,27.74,0.9161,Waning Crescent,NaT
3,2018-09-27,17:03:00,51682.7167,Daylit,18.6191,264.8042,-48.6246,64.5096,33.6896,27.75,0.9160,Waning Crescent,NaT
4,2018-09-27,17:04:00,51133.5536,Daylit,18.3733,264.8520,-48.4104,64.6582,33.6980,27.75,0.9160,Waning Crescent,NaT
5,2018-09-27,17:05:00,50582.3788,Daylit,18.1275,264.8995,-48.1959,64.8054,33.7063,27.76,0.9159,Waning Crescent,NaT
6,2018-09-27,17:06:00,50029.1795,Daylit,17.8817,264.9469,-47.9811,64.9511,33.7147,27.77,0.9159,Waning Crescent,NaT
7,2018-09-27,17:07:00,49473.9418,Daylit,17.6359,264.9942,-47.7661,65.0954,33.7231,27.78,0.9159,Waning Crescent,NaT
8,2018-09-27,17:08:00,48916.6517,Daylit,17.3900,265.0412,-47.5509,65.2383,33.7314,27.78,0.9158,Waning Crescent,NaT
9,2018-09-27,17:09:00,48357.2942,Daylit,17.1441,265.0881,-47.3354,65.3799,33.7398,27.79,0.9158,Waning Crescent,NaT
